# titanic survival lr svm

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load Titanic dataset
url = 'https://raw.githubusercontent.com/tkacha467/end-to-end-ml-projects/main/titanic.csv'
try:
    df = pd.read_csv(url)
except:
    # Fallback: use seaborn's built-in titanic
    import seaborn as sns
    df = sns.load_dataset('titanic')

print("Shape:", df.shape)
print(df.head())

In [ ]:
# Data Preprocessing
# Keep relevant columns
cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived']
# Try both column naming styles
if 'survived' in df.columns:
    df = df.rename(columns={'survived':'Survived','pclass':'Pclass','sex':'Sex',
                             'age':'Age','sibsp':'SibSp','parch':'Parch','fare':'Fare'})

df = df[['Pclass','Sex','Age','SibSp','Parch','Fare','Survived']].dropna()

# Encode Sex
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("After preprocessing:")
print(df.shape)
print(df['Survived'].value_counts())

In [ ]:
# Split Features and Target
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_lr))

In [ ]:
# SVM
svm = SVC(kernel='rbf', C=1, gamma='scale')
svm.fit(X_train_s, y_train)
y_pred_svm = svm.predict(X_test_s)

print("=== SVM (RBF Kernel) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))

In [ ]:
# Comparison
print("\n=== Model Comparison ===")
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"SVM Accuracy:                 {accuracy_score(y_test, y_pred_svm):.4f}")

# Confusion matrix heatmaps
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_pred, title in zip(axes, [y_pred_lr, y_pred_svm], ['Logistic Regression', 'SVM']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{title} - Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()